# Notebook 8: Explainability with GNNExplainer

**Objective:** Understand *why* the GNN predicts a transaction as fraudulent by identifying the most important neighboring nodes, edges, and input features using GNNExplainer.

In [1]:
import sys
sys.path.append('../src')

import torch
import matplotlib.pyplot as plt
import numpy as np

from graph_builder import load_graph
from models import GAT
from explain import get_explainer, explain_node, plot_explanation_subgraph, top_k_node_features
from utils import set_seed, get_device, load_model

set_seed(42)
device = get_device()
print('Device:', device)
%matplotlib inline

Device: cpu


In [2]:
data  = load_graph('../data/processed')
model = GAT(data.num_features, 32, 2, heads=4)
model = load_model(model, '../models/gat_model.pt', device)
model.eval()
print('Model loaded.')

Model loaded.


## 1. Find High-Confidence Fraud Predictions

In [3]:
data = data.to(device)

with torch.no_grad():
    out  = model(data.x, data.edge_index)
    prob = torch.softmax(out, dim=1)[:, 1].cpu()
    pred = out.argmax(dim=1).cpu()

test_idx = data.test_mask.nonzero(as_tuple=True)[0]
fraud_test_idx = test_idx[(pred[test_idx] == 1) & (data.y[test_idx] == 1)]
print(f'Correctly predicted fraud nodes in test set: {len(fraud_test_idx)}')

# Pick the node with highest fraud probability
top_fraud_node = fraud_test_idx[prob[fraud_test_idx].argmax()].item()
print(f'Explaining node index: {top_fraud_node} (fraud prob: {prob[top_fraud_node]:.4f})')

Correctly predicted fraud nodes in test set: 865
Explaining node index: 7929 (fraud prob: 0.9994)


## 2. Build Explainer and Generate Explanation

In [4]:
# GNNExplainer needs x and edge_index separately — same signature as model.forward
data_cpu = data.cpu()
model_cpu = model.cpu()

explainer   = get_explainer(model_cpu, epochs=50)   # 50 epochs gives good quality explanations
explanation = explain_node(explainer, data_cpu, top_fraud_node)
print(explanation)

Explanation(node_mask=[46564, 166], edge_mask=[36624], prediction=[46564, 2], target=[46564], index=[1], x=[46564, 166], edge_index=[2, 36624])


## 3. Visualise Explanation Subgraph

In [5]:
plot_explanation_subgraph(
    explanation, top_fraud_node,
    title=f'Explanation Subgraph for Fraud Node {top_fraud_node}',
    save_path='../results/figures/explanation_subgraph.png'
)

Explanation subgraph: 53 nodes, 30 edges


c:\Users\anmol\OneDrive\Desktop\GraphShield Explainable Graph Neural Network for Financial Fraud Detection\notebooks\../src\explain.py:55: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()


## 4. Top Important Features

In [6]:
print('Top 10 important features for the fraud prediction:')
top_features = top_k_node_features(explanation, k=10)

Top 10 important features for the fraud prediction:
  Feature_70: 0.0000
  Feature_88: 0.0000
  Feature_64: 0.0000
  Feature_38: 0.0000
  Feature_128: 0.0000
  Feature_62: 0.0000
  Feature_134: 0.0000
  Feature_113: 0.0000
  Feature_0: 0.0000
  Feature_76: 0.0000


In [7]:
feat_names = [f[0] for f in top_features]
feat_scores= [f[1] for f in top_features]

plt.figure(figsize=(8, 4))
plt.barh(feat_names[::-1], feat_scores[::-1], color='crimson')
plt.xlabel('Importance Score')
plt.title(f'Top 10 Node Feature Importances (Node {top_fraud_node})')
plt.tight_layout()
plt.savefig('../results/figures/feature_importance.png', dpi=150)
plt.close()

## 5. Explain Multiple Fraud Nodes

In [8]:
sample_nodes = fraud_test_idx[:5].tolist()
print('Fraud probability for sampled nodes:')
for nidx in sample_nodes:
    print(f'  Node {nidx}: fraud_prob={prob[nidx]:.4f}')

Fraud probability for sampled nodes:
  Node 1387: fraud_prob=0.6268
  Node 1465: fraud_prob=0.6879
  Node 1921: fraud_prob=0.5822
  Node 3088: fraud_prob=0.6377
  Node 3166: fraud_prob=0.6522
